![Texto alternativo](https://github.com/evmpython/Minicurso_queimadas_UNIFEI_INPE_2026/blob/main/04_logos/banner_queimadas.png?raw=true)

🎯**Objetivo:**


**Processar e visualizar** arquivos diários de focos de queimadas do satélite **AQUA_M-T** sobre o Brasil utilizando dados gratuitos do Programa Queimadas do INPE.


---


🧑**Palestrante/Tutor**

Dr. Guilherme Martins - Nottus Meteorologia

✉ guilherme.martins@nottus.com.br | jgmsantos@gmail.com

🌎 https://github.com/jgmsantos

🌎 https://guilherme.readthedocs.io/en/latest/


---


✅**Aulas ministradas**

- Aula_1_ex01: https://colab.research.google.com/drive/1mznslPhwRJlc75Vc2oJM-JR0g3M0K9b5?usp=sharing

---

📚**Material de apoio sobre Python**

https://guilherme.readthedocs.io/en/latest/pages/tutoriais/python.html

---

🎲**Dados utilizados no formato csv**
- Focos diários:
  - https://dataserver-coids.inpe.br/queimadas/queimadas/focos/csv/diario/Brasil


---

✅**Atividades a serem desenvolvidas**
1. Aprender a importar bibliotecas
1. Montar o drive para ler/salvar arquivos
1. Acessar os dados anuais e mensais de focos
1. Explorar os dados
1. Concatenar os arquivos
1. Processar e visualizar por estado
   - Filtrar datas
   - Filtrar área de interesse (município, estado ou bioma)
   - Calcular algumas métricas
1. Processar e visualizar o dado mensal
   -. Calcular acumulado mensal
   -. Calcular média mensal
   -. Heatmap
1. Processar e visualizar o dado anual
   - Calcular o acumulado anual
   - Calcular porcentagem em relação à média
   - Calcular a variação percentual do ano corrente em relação ao ano anterior

---

❗**Importante**

- Necessário possuir uma conta do Gmail.
- Salvar este código no seu Google Drive. Basta clicar em **Arquivo** (canto superior esquerdo) e depois em **Salvar uma cópia no drive** e fazer o login numa conta Google.
---

## Instalar bibliotecas

In [ ]:
# Instala o geobr para obter o mapa com os estados do Brasil.
!pip install geobr

# Mais informações sobre a biblioteca geobr.
# https://github.com/ipea/geobr

# Instala o leafmap para gerar o mapa interativo.
!pip install leafmap

# Importando bibliotecas

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import datetime as dt # Biblioteca para manipulação de data.
import geopandas as gpd # Facilita a manipulação de dados geoespaciais.
import os # Para criar diretórios para salvar as figuras.
import matplotlib.cm as cm # Biblioteca para criar mapa de cores.
import leafmap # Importação da biblioteca leafmap.
import geobr # Baixa conjuntos de dados espaciais oficiais do Brasil.

# Montar o drive para ler/salvar arquivos que estão no seu Google Drive.
# Não precisa alterar nada, é assim mesmo.
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Local para salvar as figuras.
# ALTERAR AQUI APONTANDO PARA A PASTA "figuras".
diretorio_figuras = "/content/drive/MyDrive/cursos/queimadas_2026/scripts/figuras"

# Cria o diretório acima caso ele não exista.
os.makedirs(diretorio_figuras, exist_ok=True)

# Abertura dos arquivos

In [ ]:
# URL que contém os dados diários de focos de queimadas detectado somente no Brasil.
# Neste link estão TODOS os satélites.
url = "https://dataserver-coids.inpe.br/queimadas/queimadas/focos/csv/diario/Brasil"

# Número de arquivos que estão na URL acima.
numero_arquivos_csv = 32

# Data de hoje.
hoje = dt.date.today()

# Cria uma lista ("[]") vazia para unir os arquivos csv.
lista_dataframes = []

# Total de 16 colunas.

# id	lat	lon	data_hora_gmt	satelite	municipio	estado	pais	municipio_id	estado_id	pais_id	numero_dias_sem_chuva	precipitacao	risco_fogo	bioma	frp

# Filtrar somente o que interessa usando "usecols".

# Definir a coluna "data_hora_gmt" como Datetime para realizar cálculos.
colunas_interesse = ["lat", "lon", "data_hora_gmt", "satelite", "municipio", "estado", "bioma"]

# Iteração ao longo dos dias.
for i in range(numero_arquivos_csv):

    dia = hoje - dt.timedelta(days=i) # Formato da data no estilo YYYY-MM-DD.

    data_str = dia.strftime("%Y%m%d") # Formato da data no estilo YYYYMMDD.

    print(f"Processando o dia: {data_str}")

    nome_arquivo = f"focos_diario_br_{data_str}.csv" # Nome do arquivo: focos_diario_br_YYYYMMDD.csv.

    url_arquivo = f"{url}/{nome_arquivo}" # URL completa do arquivo.

    # Bloco para checar o total de registros em cada arquivo.
    try:
      df = pd.read_csv(url_arquivo, usecols=colunas_interesse, parse_dates=["data_hora_gmt"])
      if df.empty:
        print(f"  {data_str}: arquivo sem registros.")
      else:
        lista_dataframes.append(df) # Lista de DataFrames.
        print(f"  {data_str}: {len(df)} registros.")
    except Exception as e:
      print(f"Erro ao ler {nome_arquivo}: {e}")

# Concatena a lista de DataFrames em um único DataFrame.
if lista_dataframes:
    df_total = pd.concat(lista_dataframes, ignore_index=True)
    print(f"\nTotal de registros: {len(df_total):,}")
else:
    print("\nNenhum registro encontrado.")

In [ ]:
# Cria uma cópia do df_total.
df = df_total.copy()

# Visualiza o DataFrame.
df

# Ordenar as datas cronologicamente

In [ ]:
# Ordena as datas (da mais antiga para a mais nova) usando a coluna "data_hora_gmt".
# Nota-se a coluna do index que não inicia em zero.
# Solução: resetar o index.
df = df.sort_values("data_hora_gmt")

# Visualiza o DataFrame.
df

# Resetando o index

In [ ]:
# Reseta o index (reinicializar).
df = df.reset_index(drop=True)

# Visualiza o DataFrame com index reinicializado.
df

# Checando o tipo de dado

In [ ]:
# Checa o tipo de dado do "df".
df.info()

# Renomeando colunas

In [ ]:
# Visualiza o nome das colunas do "df".
df.columns

In [ ]:
# Renomeia as colunas para nomes definidos pelo usuário.
df.columns = ["Lat", "Lon", "Data", "Satelite", "Municipio", "Estado", "Bioma"]

df

# Processamento e visualização

## Qual pergunta queremos responder?

Temos algumas **possibilidades**:

- Filtrar por:
   - satélites
   - municípios
   - estados
   - biomas

O que podemos fazer?
 - Checar apenas uma das informações acima ou realizar comparações entre elas.

## Satélites disponíveis

In [ ]:
# Lista os satélites que estão presente no DataFrame.
df["Satelite"].unique()

In [ ]:
# Total de satélites que estão presente no DataFrame.
total_satelites = len(df["Satelite"].unique())

print(f"Total de satélites: {total_satelites}")

In [ ]:
# Filtro para selecionar o satélite de interesse.
nome_satelite = "AQUA_M-T"

# O operador "==" é usado como operador de igualdade lógica. É usado para comparar igualdade.
# Compara o que está do lado esquerdo com o que está do lado direito, se foram iguais,
# recebe True, caso contrário, False.
filtro_satelite = df["Satelite"] == nome_satelite # Retorna um booleano (True ou False).

# Armazena o resultado em uma nova variável (DataFrame).
df_satelite = df[filtro_satelite]

# Visualiza o DataFrame após a seleção do satélite.
df_satelite

In [ ]:
# Reinicializa o index.
df_satelite = df_satelite.reset_index(drop=True)

# Visualiza o DataFrame após a seleção do satélite.
df_satelite

**Pensando aqui! O meu DataFrame possui datas faltantes?**


## Mostrar as data disponíveis

In [ ]:
# Extrai as datas que não se repetem.
dias_disponiveis = df_satelite["Data"].dt.date.unique() # O formato aqui não é amigável de se ver.

print(f"Dias disponíveis para o satélite {nome_satelite}.")

# Mostra na tela as datas.
for dia in sorted(dias_disponiveis):
    print(dia)

## Mostrar as datas faltantes

In [ ]:
# Gera a lista completa (conjunto) de datas esperadas.
datas_esperadas = set()

for i in range(numero_arquivos_csv): # numero_arquivos_csv = 32, definido no começo do script.
    dia = hoje - dt.timedelta(days=i)
    datas_esperadas.add(dia) # Datas completas, sem ausência de dias.

# Converte dias_disponiveis para um set (conjunto) para comparação eficiente.
dias_presentes = set(dias_disponiveis) # dias_disponiveis foi obtido anteriormente.

# Encontra as datas faltantes.
datas_faltantes = sorted(list(datas_esperadas - dias_presentes))

print(f"Datas faltantes para o satélite {nome_satelite}.")

if datas_faltantes:
    for data in datas_faltantes:
        print(data)
else:
    print(f"Nenhuma data faltante encontrada para o satélite {nome_satelite}.")

O que já temos preparado:

- satélite selecionado (AQUA_M-T) e dados organizados cronologicamente.
- sabemos que há dias faltantes.

Com essas informações já é possível fazer um plot dos focos de queimadas.

## Plot estático dos focos de queimadas

Neste ponto, temos todos os dias disponíveis que possuem focos.

In [ ]:
# Nome das colunas do DataFrame.
df_satelite.columns

In [ ]:
# Converte os pontos do df_satelite (DataFrame) para geometry.

# Lon e Lat são as colunas do df_satelite.

# Forma 1:
#gdf = gpd.GeoDataFrame(df_satelite, geometry=gpd.points_from_xy(df_satelite["Lon"], df_satelite["Lat"]), crs="epsg:4326")

# Forma 2:
gdf = gpd.GeoDataFrame(df_satelite, geometry=gpd.points_from_xy(df_satelite.Lon, df_satelite.Lat), crs="epsg:4326")

# Visualiza o GeoDataFrame.
gdf

In [ ]:
# Plot básico de TODOS os dias do satélite AQUA_M-T.
gdf.plot()

O próximo passo consiste em fazer uma figura com todos os focos utilizando o shapefile que contém todos os estados do Brasil.

In [ ]:
# Leiura do GeoDataFrame com todos os estados brasileiros.
shapefile_brasil = geobr.read_state(year=2025)

# Visualiza o GeoDataFrame.
shapefile_brasil.head()

In [ ]:
# Plot básico do shapefile.
shapefile_brasil.plot()

O próximo passo consiste em checar se gdf e shapefile_brasil possuem o mesmo sistema de coordenada espacial (CRS).

In [ ]:
# Checa o Coordinate Reference System (CRS) do gdf.
gdf.crs

In [ ]:
# Checa o Coordinate Reference System (CRS) do arquivo shapefile aberto.
shapefile_brasil.crs

In [ ]:
# Certifica que ambos os shapefiles estão no mesmo Coordinate Reference System (CRS) ou sistema de referência espacial.

# Define o CRS do shapefile_brasil igual ao CRS do gdf.
shapefile_brasil = shapefile_brasil.to_crs(gdf.crs)

# CRS do shapefile após compatibilização com o gdf.
shapefile_brasil.crs

Gera o plot.

In [ ]:
# Plot de todos os focos de queimadas sobre o mapa.

# Define o tamanho da figura.
fig, ax = plt.subplots(figsize=(10, 10))

# Primeira camada: Plot do shapefile.
shapefile_brasil.plot(ax=ax, color="lightgray", edgecolor="black", legend=True)

# Segunda camada: Plot dos focos de queimadas.
gdf.plot(ax=ax, marker="o", color="red", markersize=5, label="Focos de Queimadas")

# Formatação da figura.
ax.set_title("Focos de Queimada no Brasil")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend()
plt.grid(True, linestyle="--", alpha=0.7)

# Salva a figura.
plt.savefig(f"{diretorio_figuras}/ex02-01focos_brasil_todos_dias.png", dpi=300)

# Mostra a figura.
plt.show()

# Libera memória.
plt.close()

## Plot dinâmico dos focos de queimadas

Uso da biblioteca leafmap para gerar o mapa interativo de focos.

In [ ]:
# Verifica se a biblioteca "leafmap" está instalada.
#!pip show leafmap

In [ ]:
# Instala o leafmap para gerar o mapa interativo.
#!pip install leafmap

In [ ]:
# Apenas relembrando como é o gdf.
gdf

In [ ]:
# Importação da biblioteca leafmap.
# import leafmap

# Define o ponto central do mapa e nível de zoom.
# Quanto menor o nível de zoom, mais amplo será o mapa.
# zoom = 0 a 18.
Map = leafmap.Map(center=[-15, -60], zoom=4)

# Adiciona mapa de fundo.
Map.add_basemap("SATELLITE")

# Plot dos dados sobre o mapa.
# "Lon" e "Lat" na função abaixo são as colunas do GeoDataFrame gdf.
# x e y são os parâmetros da função "add_xy_data".
# layer_name é o nome da camada que será mostrada.
Map.add_xy_data(gdf, x="Lon", y="Lat", layer_name="Focos (AQUA)")

# Inclusão do shapefile e sua formatação.
Map.add_gdf(
    shapefile_brasil,
    layer_name="Estados",
    style={
        "color": "black",  # Cor da linha do mapa do Brasil.
        "weight": 2,       # Espessura da linha do shapefile.
        "fillOpacity": 0,  # Sem transparência no mapa. 0 = transparente.
    },
)

# Exibe o mapa.
Map

## O que fizemos até aqui?

- Abertura dos arquivos diários com todos os satélites.
- Pré-processamento (ordenação de data, reinicialização do index, checagem do tipo de dado e renomeação de colunas).
- Processamento e visualização inicial dos focos usando o satélite AQUA_M-T.
  - Checagem de dias faltantes.
- Plot estático (Matplotlib) e dinâmico (leafmap) de todos os dias com focos para o Brasil.
  - Compatibilização do sistema de coordenada espacial (CRS)

> Foram verificados os focos que ocorreram no Brasil e se quisessemos escolher uma região (Estado) de interesse, como fazer?

## Definir uma região de interesse para plotar os focos

In [ ]:
# Relembrando quem é gdf.
gdf

In [ ]:
# Cópia do GeoDataFrame original para não bagunçar com as demais análise anteriores.
gdf_regiao = gdf.copy()

In [ ]:
# Relembrando quem é o shapefile_brasil.
shapefile_brasil

Será usado o shapefile_brasil para selecionar um Estado de interesse.

In [ ]:
# O operador "==" é usado como operador de igualdade lógica. É usado para comparar igualdade.
# Compara o que está do lado esquerdo com o que está do lado direito, se foram iguais,
# recebe True, caso contrário, False.

# Filtro para selecionar o estado de interesse.
filtro_estado = shapefile_brasil["name_state"] == "Tocantins" # O resultado é booleano (True ou False).

# Aplica o filtro e só retorna o que é True.
shapefile_selecionado = shapefile_brasil[filtro_estado]

# Plot básico do shapefile_selecionado.
shapefile_selecionado.plot()

In [ ]:
# Realiza a interseção espacial para mascarar os dados.

# O shapefile_selecionado é considerado uma máscara. Os focos de queimadas são
# considerados apenas dentro do domínio do shapefile_selecionado.

dado_mascarado = gpd.overlay(gdf_regiao, shapefile_selecionado, how="intersection")

# Visualiza o novo GeoDataFrame.
dado_mascarado

In [ ]:
# Plot de todos os focos de queimadas sobre o mapa.

# Define o tamanho da figura.
fig, ax = plt.subplots(figsize=(10, 10))

# Plot do shapefile.
shapefile_selecionado.plot(ax=ax, color="lightgray", edgecolor="black", legend=True)

# Plot dos focos de queimadas.
dado_mascarado.plot(ax=ax, marker="o", color="red", markersize=5, label="Focos de Queimadas")

# Formatação da figura.
ax.set_title("Focos de Queimadas em uma área definida pelo shapefile")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend()
plt.grid(True, linestyle="--", alpha=0.7)

# Salva a figura.
plt.savefig(f"{diretorio_figuras}/ex02-02focos_brasil_estado.png", dpi=300)

# Mostra a figura.
plt.show()

# Libera memória.
plt.close()

Até o momento, foram plotados todos os dias no mapa. Agora, queremos aplicar um filtro para selecionar apenas um dia.

## Selecionar um dia

In [ ]:
# Relembrando quem é o gdf.
gdf

In [ ]:
gdf.info()

In [ ]:
# Dia de interesse.
dia_escolhido = "2026-07-20" # Aqui é uma string e não DateTime.

# Filtro a ser utilizado.
filtro_data = gdf["Data"].dt.date == pd.to_datetime(dia_escolhido).date()

# Aplica o filtro para selecionar apenas a data de interesse.
df_data_foco = gdf[filtro_data]

# Visualiza o GeoDataFrame para o dia selecionado.
df_data_foco

In [ ]:
# Plot dos focos de queimadas para um dia.

# Define o tamanho da figura.
fig, ax = plt.subplots(figsize=(10, 10))

# Primeira camada: Plot do shapefile.
shapefile_brasil.plot(ax=ax, color="lightgray", edgecolor="black", legend=True)

# Segunda camada: Plot dos focos de queimadas.
df_data_foco.plot(ax=ax, marker="o", color="red", markersize=5, label="Focos de Queimadas")

# Formatação da figura.
ax.set_title(f"Focos de Queimadas no Brasil no dia {dia_escolhido}")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend()
plt.grid(True, linestyle="--", alpha=0.7)

# Salva a figura.
plt.savefig(f"{diretorio_figuras}/ex02-03focos_brasil_1dia.png", dpi=300)

# Mostra a figura.
plt.show()

# Libera memória.
plt.close()

Há também a possibilidade de seleção de um intervalo de datas.

## Selecionar um intervalo de dadas

In [ ]:
data_inicio = "2026-07-15" # Data de início de interesse.
data_fim = "2026-07-20" # Data de fim de interesse.

# Filtro para o intervalo de datas.
# O operador ">=" significa maior ou igual a.
# O operador "<=" significa menor ou igual a.
# O "&" é o operador AND lógico do Pandas (elemento por elemento).
# Ele exige que as duas condições sejam verdadeiras ao mesmo tempo.
filtro_data = (gdf["Data"].dt.date >= pd.to_datetime(data_inicio).date()) & \
              (gdf["Data"].dt.date <= pd.to_datetime(data_fim).date())

# Aplicação do filtro para o intervalo de datas de interesse.
df_data_foco = gdf[filtro_data]

# Visualiza o DataFrame para o intervalo de datas selecionado.
df_data_foco

In [ ]:
# Lista com as datas que não se repetem.
# Será usada para criar a escala de cores que depende do número de dias.
unique_dates = df_data_foco["Data"].dt.date.unique()
unique_dates_sorted = sorted(unique_dates)

# Cria uma tabela de cores (viridis) de acordo com a quantidade de dias.
# https://matplotlib.org/stable/users/explain/colors/colormaps.html
# colors é umm objeto que pode ser chamado como uma função.
colors = cm.get_cmap("viridis", len(unique_dates_sorted))

# Se quiser usar cores definidas.
# Na linha do plot, alterar de color=colors(i) para color=colors[i].
# Descomentar a linha abaixo e comentar colors de cima.
# colors = ["red", "black", "orange"]

# Configuração do plot.
fig, ax = plt.subplots(figsize=(10, 10))

# Primeira camada: Plot do shapefile do Brasil.
shapefile_brasil.plot(ax=ax, color="lightgray", edgecolor="black")

# Segunda camada: Plot dos focos para cada dia, cada dia tem uma cor diferente.
for i, date in enumerate(unique_dates_sorted):
    df_day = df_data_foco[df_data_foco["Data"].dt.date == date]
    if not df_day.empty:
        df_day.plot(ax=ax, marker="o", color=colors(i), markersize=5, label=str(date))

# Formatação do plot.
ax.set_title(f"Focos de Queimadas no Brasil entre {data_inicio} e {data_fim}")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend(title="Data", bbox_to_anchor=(0.8, 0.3), loc="upper left") # Posição da legenda.
plt.grid(True, linestyle="--", alpha=0.7)
plt.tight_layout()

# Salva a figura.
plt.savefig(f"{diretorio_figuras}/ex02-04focos_brasil_intervalo_dias.png", dpi=300)

# Mostra a figura.
plt.show()

# Libera memória.
plt.close()

## O que fizemos até aqui?

- Cópia do GeoDataFrame.
- A partir do shapefile selecionamos um estado de interesse.
- Interseção espacial do GeoDataFrame e do shapefile selecionado.
- Plot de todos os focos disponíveis no estado de interesse.
- Filtragem de datas para selecionar apenas um dia para visualização.
- Filtragem de datas para selecionar um intervalo de dias para visualização.
  - Definição de cores.

## Criar um mapa temático de focos por estado brasileiro

O que faremos?
- realizar cópia do df_contagem (não quero bagunçar o original).
- realizar cópia do shapefile_brasil (não quero bagunçar o original).
- padronizar as colunas de shapefile_brasil usando o df_contagem como referência.
  - Alterar o nome da coluna do shapefile_brasil de NM_UF para Estado e deixar os nomes dos estados em maiuscúlo.
- Pulo do gato: realizar o merge do df_contagem_tematico e shapefile_brasil_tematico pela coluna Estado. Por isso, a padronização.
  - Todas as informações (estados e focos) em um único GeoDataFrame.

In [ ]:
# Apenas para relembrar como é a estrutura do gdf.
gdf

In [ ]:
 # Nome da coluna que está no DataFrame.
coluna_interesse = "Estado"

# Contabiliza a quantidade de focos.
contagem_focos = gdf.groupby(by=coluna_interesse).Estado.count()

# Cria um DataFrame para armazenar os resultados.
df_contagem = pd.DataFrame({"Estado":contagem_focos.index, "#Focos":contagem_focos.values})

# Visualiza o DataFrame.
df_contagem

In [ ]:
# Cria uma cópia do df_contagem para não bagunçar o original.
df_mapa_tematico = df_contagem.copy()

df_mapa_tematico

In [ ]:
# Apenas relembrando como é o GeoDataFrame shapefile_brasil que será utilizado.
shapefile_brasil

In [ ]:
# Cria uma cópia de shapefile_brasil para não bagunçar o original.
shapefile_brasil_tematico = shapefile_brasil.copy()

# Altera o nome da coluna de NM_UF para Estado diretamente no shapefile_brasil_tematico
# para manter o mesmo padrão do df_mapa_tematico.
shapefile_brasil_tematico.rename(columns={"name_state": "Estado"}, inplace=True)

# Altera o nome para maiuscúlo para manter o mesmo padrão da coluna Estado do df_mapa_tematico.
shapefile_brasil_tematico["Estado"] = shapefile_brasil_tematico["Estado"].str.upper()

# Visualização do GeoDataFrame.
shapefile_brasil_tematico

In [ ]:
# Pulo do gato!
# Realiza a união (merge) considerando a coluna comum nos dois (shapefile_brasil_tematico e df_mapa_tematico) que é o "Estado".
# Detalhe: O nome da coluna (Estado) e o nome dos estados TEM QUE SER IGUAIS em ambos os GeoDataFrames.
resultado = pd.merge(shapefile_brasil_tematico, df_mapa_tematico, how="left", left_on=["Estado"], right_on=["Estado"] )

# Caso tenha valor NaN (ausente) em algum dos estados, substitui este valor por zero no próprio GeoDataFrame.
resultado["#Focos"].fillna(0, inplace=True)

# Visualiza o GeoDataFrame.
resultado

In [ ]:
# Plot dos focos.
ax = resultado.plot(
    column="#Focos",
    scheme="Equal_Interval",
    cmap="viridis", # Para alterar a cor: https://matplotlib.org/stable/users/explain/colors/colormaps.html
    figsize=(16, 8),
    legend=True,
    edgecolor="black",
    legend_kwds={"loc": "lower right"}
)

# Título do mapa.
ax.set_title("Focos de Queimadas por estado brasileiro")

# Salva a figura.
plt.savefig(f"{diretorio_figuras}/ex02-05focos_mapa_tematico.png", dpi=300)

# Mostra a figura.
plt.show()

# Libera memória.
plt.close()